# 1 — Repeated LLM classification

Classifies every unique incident five times with six frozen model endpoints, then performs five additional fixed-batch reruns on a deterministic 1,000-description sample. The first experiment randomizes description order by run; the second freezes batch membership, position, and order to isolate ordinary rerun variation. Outputs are schema-constrained, validated, retried, resumable, and retained verbatim for audit.

The only private configuration is the project-root `.env`, which requires `OPENAI_API_KEY`, `GOOGLE_API_KEY`, and `DEEPINFRA_API_KEY`.

**Paper coverage:** §3.2, Table 2 (models), and the fixed-input design in Appendix A.4.

Use the pinned environment in [README.md](README.md#run-the-analysis). Run cells from top to bottom; saved inference caches allow reproduction without API keys or new requests.


In [ ]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from threading import BoundedSemaphore
from dotenv import load_dotenv
from IPython.display import display
from tqdm import tqdm
import pandas as pd
import numpy as np
import hashlib, json, os, random, time, requests

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "s0_get_data.ipynb").exists()), Path.cwd())
load_dotenv(ROOT / ".env", override=False)

def require_api_keys(providers):
    key_names = {"OpenAI": "OPENAI_API_KEY", "Google": "GOOGLE_API_KEY", "DeepInfra": "DEEPINFRA_API_KEY"}
    missing = [key_names[p] for p in sorted(providers) if not os.getenv(key_names[p])]
    if missing:
        raise RuntimeError("New inference requests need these keys in the project .env: " + ", ".join(missing))

DESCRIPTIONS_PATH = ROOT / "data" / "processed" / "descriptions.csv"
if not DESCRIPTIONS_PATH.exists():
    raise FileNotFoundError("Run s0_get_data.ipynb first.")
RESULTS = ROOT / "data" / "results"
TABLES = ROOT / "tables"
RESULTS.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

LABELS = ["negligible", "low", "medium", "high", "catastrophic"]
LABEL_VALUE = {label: i + 1 for i, label in enumerate(LABELS)}
RUNS, BATCH_SIZE, SEED = 5, 20, 20260812
MAX_WORKERS, MAX_ATTEMPTS = 96, 7
descriptions = pd.read_csv(DESCRIPTIONS_PATH).sort_values("description_id").reset_index(drop=True)


## Frozen roster and inference policy

OpenAI uses the Responses API with reasoning disabled; Gemini uses low thinking; DeepInfra models use reasoning disabled. A temperature of 0.1 is sent where supported. The DeepInfra endpoint names are stable aliases, not immutable provider snapshots, so returned model identifiers and timestamps are recorded for every batch. The cited Hugging Face revisions document the open-weight artifacts and licenses available at freeze time; DeepInfra may update an alias independently.

**Reports:** §3.2; Table 2, Panel B.


In [2]:
MODELS = [
    {
        "model": "GPT-5.6 Sol", "provider": "OpenAI", "model_id": "gpt-5.6-sol",
        "license": "proprietary", "artifact_revision": "provider endpoint",
        "endpoint": "https://api.openai.com/v1/responses", "temperature": np.nan, "reasoning": "none",
    },
    {
        "model": "Gemini 3.1 Pro", "provider": "Google", "model_id": "gemini-3.1-pro-preview",
        "license": "proprietary", "artifact_revision": "provider preview endpoint",
        "endpoint": "Google generateContent v1beta", "temperature": 0.1, "reasoning": "low",
    },
    {
        "model": "Kimi K2.6", "provider": "DeepInfra", "model_id": "moonshotai/Kimi-K2.6",
        "license": "Modified MIT", "artifact_revision": "7eb5002f6aadc958aed6a9177b7ed26bb94011bb",
        "endpoint": "DeepInfra OpenAI-compatible chat", "temperature": 0.1, "reasoning": "none",
    },
    {
        "model": "DeepSeek V4 Pro", "provider": "DeepInfra", "model_id": "deepseek-ai/DeepSeek-V4-Pro",
        "license": "MIT", "artifact_revision": "b5968e9190ef611bbf34a7229255be88a0e937c1",
        "endpoint": "DeepInfra OpenAI-compatible chat", "temperature": 0.1, "reasoning": "none",
    },
    {
        "model": "Qwen3.5 397B", "provider": "DeepInfra", "model_id": "Qwen/Qwen3.5-397B-A17B",
        "license": "Apache-2.0", "artifact_revision": "8472618112abcbd45acbcdc58436aff4233c23f7",
        "endpoint": "DeepInfra OpenAI-compatible chat", "temperature": 0.1, "reasoning": "none",
    },
    {
        "model": "GLM-5.2", "provider": "DeepInfra", "model_id": "zai-org/GLM-5.2",
        "license": "MIT", "artifact_revision": "b4734de4facf877f85769a911abafc5283eab3d9",
        "endpoint": "DeepInfra OpenAI-compatible chat", "temperature": 0.1, "reasoning": "none",
    },
]
roster = pd.DataFrame(MODELS)
roster["freeze_date"] = "2026-08-12"
roster["runs"] = RUNS
roster["batch_size"] = BATCH_SIZE
roster.to_csv(TABLES / "model_roster.csv", index=False)
display(roster[["model", "provider", "model_id", "license", "temperature", "reasoning"]])


,model,provider,model_id,license,temperature,reasoning
0,GPT-5.6 Sol,OpenAI,gpt-5.6-sol,proprietary,NaN,none
1,Gemini 3.1 Pro,Google,gemini-3.1-pro-preview,proprietary,0.1,low
2,Kimi K2.6,DeepInfra,moonshotai/Kimi-K2.6,Modified MIT,0.1,none
3,DeepSeek V4 Pro,DeepInfra,deepseek-ai/DeepSeek-V4-Pro,MIT,0.1,none
4,Qwen3.5 397B,DeepInfra,Qwen/Qwen3.5-397B-A17B,Apache-2.0,0.1,none
5,GLM-5.2,DeepInfra,zai-org/GLM-5.2,MIT,0.1,none


In [3]:
def make_prompt(batch):
    lines = "\n".join(f"{i + 1}. {text}" for i, text in enumerate(batch["Description"]))
    return (
        f"The following are descriptions of {len(batch)} property insurance incidents:\n{lines}\n"
        "You are an actuarial assistant assigned to classify these insurance incidents into one of five "
        "categories: negligible, low, medium, high, catastrophic. "
        f"Return a JSON object with a labels array containing exactly {len(batch)} classification names "
        "in the same order. Do not include explanations."
    )

def make_schema(n):
    return {
        "type": "object",
        "properties": {
            "labels": {
                "type": "array", "items": {"type": "string", "enum": LABELS},
                "minItems": n, "maxItems": n,
            }
        },
        "required": ["labels"],
        "additionalProperties": False,
    }

def extract_json(text):
    text = (text or "").strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
    return json.loads(text)

def validate_labels(text, n):
    values = extract_json(text).get("labels")
    if not isinstance(values, list) or len(values) != n:
        raise ValueError(f"Expected {n} labels, received {0 if values is None else len(values)}")
    normalized = [str(value).strip().lower() for value in values]
    invalid = [value for value in normalized if value not in LABEL_VALUE]
    if invalid:
        raise ValueError(f"Invalid labels: {invalid}")
    return normalized


### Provider adapters

Use the same output schema and validate positional labels for every provider.


In [4]:
def call_openai(model_id, prompt, n):
    payload = {
        "model": model_id,
        "input": [{"role": "user", "content": prompt}],
        "reasoning": {"effort": "none"},
        "text": {"format": {"type": "json_schema", "name": "risk_labels", "strict": True, "schema": make_schema(n)}},
        "max_output_tokens": 400,
    }
    response = requests.post(
        "https://api.openai.com/v1/responses",
        headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}", "Content-Type": "application/json"},
        json=payload, timeout=180,
    )
    response.raise_for_status()
    data = response.json()
    text = "".join(
        item.get("text", "")
        for output in data.get("output", [])
        for item in output.get("content", [])
        if item.get("type") == "output_text"
    )
    return data, text, data.get("model", model_id), data.get("usage", {})

def call_google(model_id, prompt, n):
    payload = {
        "contents": [{"role": "user", "parts": [{"text": prompt}]}],
        "generationConfig": {
            "temperature": 0.1,
            "responseMimeType": "application/json",
            "responseJsonSchema": make_schema(n),
            "thinkingConfig": {"thinkingLevel": "LOW"},
        },
    }
    response = requests.post(
        f"https://generativelanguage.googleapis.com/v1beta/models/{model_id}:generateContent",
        headers={"x-goog-api-key": os.environ["GOOGLE_API_KEY"], "Content-Type": "application/json"}, json=payload, timeout=180,
    )
    response.raise_for_status()
    data = response.json()
    text = data["candidates"][0]["content"]["parts"][0]["text"]
    return data, text, data.get("modelVersion", model_id), data.get("usageMetadata", {})

def call_deepinfra(model_id, prompt, n):
    payload = {
        "model": model_id,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
        "reasoning_effort": "none",
        "max_tokens": 400,
        "response_format": {
            "type": "json_schema",
            "json_schema": {"name": "risk_labels", "strict": True, "schema": make_schema(n)},
        },
    }
    response = requests.post(
        "https://api.deepinfra.com/v1/openai/chat/completions",
        headers={"Authorization": f"Bearer {os.environ['DEEPINFRA_API_KEY']}", "Content-Type": "application/json"},
        json=payload, timeout=180,
    )
    response.raise_for_status()
    data = response.json()
    text = data["choices"][0]["message"]["content"]
    return data, text, data.get("model", model_id), data.get("usage", {})

CALLERS = {"OpenAI": call_openai, "Google": call_google, "DeepInfra": call_deepinfra}
PROVIDER_SLOTS = {"OpenAI": BoundedSemaphore(40), "Google": BoundedSemaphore(4), "DeepInfra": BoundedSemaphore(48)}


### Validated request and retry policy

Technical retries retain the same prompt and settings; saved records preserve raw responses.


In [5]:
def classify_task(task):
    model = task["config"]
    prompt = make_prompt(task["batch"])
    started = time.perf_counter()
    last_error = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            with PROVIDER_SLOTS[model["provider"]]:
                raw, text, returned_model, usage = CALLERS[model["provider"]](
                    model["model_id"], prompt, len(task["batch"])
                )
            labels = validate_labels(text, len(task["batch"]))
            elapsed = time.perf_counter() - started
            rows = []
            for position, ((_, item), label) in enumerate(zip(task["batch"].iterrows(), labels)):
                rows.append({
                    "description_id": item["description_id"], "model": model["model"],
                    "provider": model["provider"], "model_id": model["model_id"],
                    "returned_model": returned_model, "run": task["run"], "batch_id": task["batch_id"],
                    "position_in_batch": position, "label": label, "label_value": LABEL_VALUE[label],
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                })
            audit = {
                "request_key": task["request_key"], "model": model["model"], "provider": model["provider"],
                "requested_model": model["model_id"], "returned_model": returned_model,
                "run": task["run"], "batch_id": task["batch_id"], "attempt": attempt,
                "timestamp_utc": datetime.now(timezone.utc).isoformat(), "latency_seconds": elapsed,
                "description_ids": task["batch"]["description_id"].tolist(),
                "prompt_sha256": hashlib.sha256(prompt.encode()).hexdigest(),
                "settings": {"temperature": model["temperature"], "reasoning": model["reasoning"], "batch_size": len(task["batch"])},
                "usage": usage, "raw_response": raw,
            }
            return rows, audit
        except Exception as error:
            last_error = error
            if attempt < MAX_ATTEMPTS:
                retry_after = getattr(getattr(error, "response", None), "headers", {}).get("retry-after", 0)
                try:
                    retry_after = float(retry_after)
                except (TypeError, ValueError):
                    retry_after = 0
                time.sleep(max(retry_after, min(90, 5 * (2 ** (attempt - 1)) + random.random())))
    raise RuntimeError(f"{task['request_key']} failed after {MAX_ATTEMPTS} attempts: {last_error}")


## Compatibility smoke test

A new six-call pilot validates live credentials, model availability, structured-output behavior, and the common parser before the full experiment begins. Cached reproduction displays the saved pilot and makes no live calls. The pilot does not enter the research dataset.

**Reports:** §3.2; compatibility audit only, excluded from research results.


In [6]:
pilot_path = TABLES / "api_smoke_test.csv"
if pilot_path.exists():
    pilot = pd.read_csv(pilot_path)
    print("Using the previously completed compatibility smoke test.")
else:
    require_api_keys({config["provider"] for config in MODELS})
    pilot_batch = descriptions.iloc[:5].copy()
    pilot_rows = []
    for config in tqdm(MODELS, desc="API smoke test"):
        task = {"config": config, "run": 0, "batch_id": 0, "batch": pilot_batch, "request_key": f"pilot::{config['model_id']}"}
        rows, audit = classify_task(task)
        pilot_rows.extend(rows)
    pilot = pd.DataFrame(pilot_rows)
    pilot.to_csv(pilot_path, index=False)
display(pilot.pivot(index="description_id", columns="model", values="label"))


Using the previously completed compatibility smoke test.


model,DeepSeek V4 Pro,GLM-5.2,GPT-5.6 Sol,Gemini 3.1 Pro,Kimi K2.6,Qwen3.5 397B
description_id,,,,,,
d_00032fbe17cf579f,low,negligible,low,negligible,low,negligible
d_0003755686a0409c,medium,medium,medium,medium,medium,low
d_001c0c7d23a3d5f8,low,low,low,negligible,low,medium
d_001c9a661e2df3b9,high,high,high,high,medium,high
d_00281de012e7816a,negligible,negligible,negligible,low,negligible,catastrophic


## Full repeated experiment

A clean run creates 7,050 logical requests: 7,020 contain 20 descriptions and 30 final batches contain 10. OpenAI and DeepInfra requests execute concurrently; Gemini logical requests are submitted through Google's asynchronous Batch API, which preserves each logical request while using a separate large-scale evaluation quota. Successful results are appended immediately. Re-runs resume standard requests and an existing Gemini batch job without duplicate submission.

**Reports:** §3.2; 140,700 main-experiment labels.


In [7]:
# Construct a shared random order for each run, then reuse its batches for every model.
tasks = []
for run in range(1, RUNS + 1):
    ordered = descriptions.sample(frac=1, random_state=SEED + run).reset_index(drop=True)
    batches = [ordered.iloc[start:start + BATCH_SIZE].copy() for start in range(0, len(ordered), BATCH_SIZE)]
    for config in MODELS:
        for batch_id, batch in enumerate(batches):
            tasks.append({
                "config": config, "run": run, "batch_id": batch_id, "batch": batch,
                "request_key": f"{config['model_id']}::run{run:02d}::batch{batch_id:04d}",
            })

labels_path = RESULTS / "classifications.csv"
audit_path = RESULTS / "responses.jsonl"
existing = pd.read_csv(labels_path) if labels_path.exists() else pd.DataFrame()

# Index saved batches once for fast cached reproduction.
cached_batches = ({} if existing.empty else
    existing.groupby(["model_id", "run", "batch_id"])["description_id"].agg(frozenset).to_dict())

def task_is_complete(task):
    key = (task["config"]["model_id"], task["run"], task["batch_id"])
    return cached_batches.get(key) == frozenset(task["batch"]["description_id"])

def persist(rows, audit):
    with audit_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(audit, ensure_ascii=False, default=str) + "\n")
    pd.DataFrame(rows).to_csv(labels_path, mode="a", header=not labels_path.exists(), index=False)

pending = [task for task in tasks if not task_is_complete(task)]
random.Random(SEED).shuffle(pending)
require_api_keys({task["config"]["provider"] for task in pending})
google_pending = [task for task in pending if task["config"]["provider"] == "Google"]
standard_pending = [task for task in pending if task["config"]["provider"] != "Google"]
print(
    f"{len(tasks):,} total logical batches; {len(tasks) - len(pending):,} complete; "
    f"{len(standard_pending):,} standard and {len(google_pending):,} Gemini batches pending."
)


7,050 total logical batches; 7,050 complete; 0 standard and 0 Gemini batches pending.


### Submit or resume Gemini batches

Only uncached requests are submitted; original logical batch boundaries remain intact.


In [8]:
# Submit one resumable Gemini batch job. Each inline entry remains one original 20-description request.
google_job_path = RESULTS / "gemini_batch_job.json"
google_task_lookup = {task["request_key"]: task for task in tasks if task["config"]["provider"] == "Google"}
google_job_name = None
if google_pending:
    pending_keys = {task["request_key"] for task in google_pending}
    saved_job = json.loads(google_job_path.read_text()) if google_job_path.exists() else None
    if saved_job and pending_keys.issubset(set(saved_job["request_keys"])):
        google_job_name = saved_job["name"]
        print(f"Resuming Gemini batch job {google_job_name}.")
    else:
        inline_requests = []
        for task in google_pending:
            prompt = make_prompt(task["batch"])
            request = {
                "contents": [{"role": "user", "parts": [{"text": prompt}]}],
                "generationConfig": {
                    "temperature": 0.1,
                    "responseMimeType": "application/json",
                    "responseJsonSchema": make_schema(len(task["batch"])),
                    "thinkingConfig": {"thinkingLevel": "LOW"},
                },
            }
            inline_requests.append({"request": request, "metadata": {"key": task["request_key"]}})
        payload = {
            "batch": {
                "display_name": f"llm-var-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}",
                "input_config": {"requests": {"requests": inline_requests}},
            }
        }
        payload_bytes = len(json.dumps(payload).encode())
        if payload_bytes >= 20_000_000:
            raise ValueError(f"Gemini inline batch is {payload_bytes / 1e6:.1f} MB; use file input.")
        response = requests.post(
            "https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:batchGenerateContent",
            headers={"x-goog-api-key": os.environ["GOOGLE_API_KEY"], "Content-Type": "application/json"},
            json=payload, timeout=180,
        )
        response.raise_for_status()
        created = response.json()
        google_job_name = created["name"]
        saved_job = {
            "name": google_job_name,
            "request_keys": [task["request_key"] for task in google_pending],
            "created_response": created,
        }
        google_job_path.write_text(json.dumps(saved_job, indent=2))
        print(f"Submitted {len(google_pending):,} Gemini requests as {google_job_name} ({payload_bytes / 1e6:.2f} MB).")


### Execute uncached standard requests

Cached runs have no pending tasks and make no API calls.


In [9]:
# Standard providers run in independent pools so one provider cannot starve another.
failures = []
provider_workers = {"OpenAI": 40, "DeepInfra": 48}
pools = {provider: ThreadPoolExecutor(max_workers=workers) for provider, workers in provider_workers.items()}
try:
    future_to_task = {
        pools[task["config"]["provider"]].submit(classify_task, task): task
        for task in standard_pending
    }
    for future in tqdm(as_completed(future_to_task), total=len(future_to_task), desc="OpenAI + DeepInfra batches"):
        task = future_to_task[future]
        try:
            rows, audit = future.result()
            persist(rows, audit)
        except Exception as error:
            failures.append(str(error))
finally:
    for pool in pools.values():
        pool.shutdown(wait=True)



OpenAI + DeepInfra batches: 0it [00:00, ?it/s]


OpenAI + DeepInfra batches: 0it [00:00, ?it/s]

### Collect and validate main-experiment labels

Parse any pending Gemini results, then require complete, unique model/run/description coverage.


In [10]:
# Poll and parse the asynchronous Gemini job after standard work has had time to finish.
if google_job_name:
    terminal_states = {"BATCH_STATE_SUCCEEDED", "BATCH_STATE_FAILED", "BATCH_STATE_CANCELLED", "BATCH_STATE_EXPIRED"}
    while True:
        response = requests.get(
            f"https://generativelanguage.googleapis.com/v1beta/{google_job_name}",
            headers={"x-goog-api-key": os.environ["GOOGLE_API_KEY"]},
            timeout=180,
        )
        response.raise_for_status()
        operation = response.json()
        state = operation.get("metadata", {}).get("state", operation.get("state"))
        stats = operation.get("metadata", {}).get("batchStats", operation.get("batchStats", {}))
        print(f"Gemini batch state: {state}; stats: {stats}")
        if operation.get("done") or state in terminal_states:
            break
        time.sleep(15)

    if operation.get("error"):
        failures.append(f"Gemini batch failed: {operation['error']}")
    else:
        batch_result = operation.get("response", operation)
        # Long-running REST operations return responses directly; resource GETs may nest them under output.
        response_container = batch_result.get("inlinedResponses") or batch_result.get("output", {}).get("inlinedResponses", {})
        batch_output = response_container.get("inlinedResponses", [])
        if not batch_output and state != "BATCH_STATE_SUCCEEDED":
            failures.append(f"Gemini batch ended in state {state}.")
        for item in batch_output:
            request_key = item.get("metadata", {}).get("key")
            task = google_task_lookup.get(request_key)
            if task is None or task_is_complete(task):
                continue
            if item.get("error"):
                failures.append(f"{request_key}: {item['error']}")
                continue
            try:
                raw = item["response"]
                text = raw["candidates"][0]["content"]["parts"][0]["text"]
                parsed = validate_labels(text, len(task["batch"]))
                returned_model = raw.get("modelVersion", task["config"]["model_id"])
                timestamp = datetime.now(timezone.utc).isoformat()
                rows = [
                    {
                        "description_id": row.description_id, "model": task["config"]["model"],
                        "provider": "Google", "model_id": task["config"]["model_id"],
                        "returned_model": returned_model, "run": task["run"], "batch_id": task["batch_id"],
                        "position_in_batch": position, "label": label, "label_value": LABEL_VALUE[label],
                        "timestamp_utc": timestamp,
                    }
                    for position, (row, label) in enumerate(zip(task["batch"].itertuples(), parsed))
                ]
                prompt = make_prompt(task["batch"])
                audit = {
                    "request_key": request_key, "model": task["config"]["model"], "provider": "Google",
                    "requested_model": task["config"]["model_id"], "returned_model": returned_model,
                    "run": task["run"], "batch_id": task["batch_id"], "attempt": 1,
                    "timestamp_utc": timestamp, "latency_seconds": None,
                    "description_ids": task["batch"]["description_id"].tolist(),
                    "prompt_sha256": hashlib.sha256(prompt.encode()).hexdigest(),
                    "settings": {"temperature": 0.1, "reasoning": "low", "batch_size": len(task["batch"]), "service": "Batch API"},
                    "usage": raw.get("usageMetadata", {}), "raw_response": raw,
                }
                persist(rows, audit)
            except Exception as error:
                failures.append(f"{request_key}: {error}")

if failures:
    print("\n".join(failures[:20]))
    raise RuntimeError(f"{len(failures)} batches failed. Re-run this cell to retry only missing batches.")

classifications = pd.read_csv(labels_path)
key = ["description_id", "model_id", "run"]
classifications = classifications.drop_duplicates(key, keep="last").sort_values(["model", "run", "description_id"])
classifications.to_csv(labels_path, index=False)

expected = len(descriptions) * len(MODELS) * RUNS
assert len(classifications) == expected, f"Expected {expected:,} rows, found {len(classifications):,}"
assert classifications[key].duplicated().sum() == 0
assert set(classifications["label"]) == set(LABELS)
print(f"Complete: {len(classifications):,} validated classifications.")


Complete: 140,700 validated classifications.


## Fixed-batch rerun experiment

To separate ordinary rerun variation from sensitivity to neighboring claims, a deterministic sample of 1,000 descriptions is frozen into 50 batches. Every model classifies those exact batches, in the exact same order, five additional times. The prompt, schema, and inference settings are unchanged. This yields 30,000 additional classifications and permits like-for-like run-level comparisons.

**Reports:** §3.2; Appendix A.4, fixed-input design.


In [11]:
FIXED_N, FIXED_SEED = 1000, 20260813
fixed_manifest_path = RESULTS / "fixed_batch_manifest.csv"
if fixed_manifest_path.exists():
    fixed_manifest = pd.read_csv(fixed_manifest_path)
else:
    fixed_manifest = descriptions.sample(n=FIXED_N, random_state=FIXED_SEED).reset_index(drop=True)
    fixed_manifest.insert(0, "fixed_position", np.arange(len(fixed_manifest)))
    fixed_manifest["batch_id"] = fixed_manifest["fixed_position"] // BATCH_SIZE
    fixed_manifest["position_in_batch"] = fixed_manifest["fixed_position"] % BATCH_SIZE
    fixed_manifest.to_csv(fixed_manifest_path, index=False)

assert len(fixed_manifest) == FIXED_N
assert fixed_manifest["description_id"].nunique() == FIXED_N
assert fixed_manifest.groupby("batch_id")["description_id"].size().eq(BATCH_SIZE).all()
fixed_batches = [
    batch[["description_id", "Description"]].copy()
    for _, batch in fixed_manifest.sort_values("fixed_position").groupby("batch_id", sort=True)
]

fixed_tasks = []
for run in range(1, RUNS + 1):
    for config in MODELS:
        for batch_id, batch in enumerate(fixed_batches):
            fixed_tasks.append({
                "config": config, "run": run, "batch_id": batch_id, "batch": batch,
                "request_key": f"fixed::{config['model_id']}::run{run:02d}::batch{batch_id:04d}",
            })

fixed_labels_path = RESULTS / "classifications_fixed.csv"
fixed_audit_path = RESULTS / "responses_fixed.jsonl"
fixed_existing = pd.read_csv(fixed_labels_path) if fixed_labels_path.exists() else pd.DataFrame()

# Index saved batches once for fast cached reproduction.
cached_fixed_batches = ({} if fixed_existing.empty else
    fixed_existing.groupby(["model_id", "run", "batch_id"])["description_id"].agg(frozenset).to_dict())

def fixed_task_is_complete(task):
    key = (task["config"]["model_id"], task["run"], task["batch_id"])
    return cached_fixed_batches.get(key) == frozenset(task["batch"]["description_id"])

def persist_fixed(rows, audit):
    audit["experiment"] = "fixed_batch"
    for row in rows:
        row["experiment"] = "fixed_batch"
    with fixed_audit_path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(audit, ensure_ascii=False, default=str) + "\n")
    pd.DataFrame(rows).to_csv(
        fixed_labels_path, mode="a", header=not fixed_labels_path.exists(), index=False
    )

fixed_pending = [task for task in fixed_tasks if not fixed_task_is_complete(task)]
random.Random(FIXED_SEED).shuffle(fixed_pending)
require_api_keys({task["config"]["provider"] for task in fixed_pending})
fixed_google_pending = [task for task in fixed_pending if task["config"]["provider"] == "Google"]
fixed_standard_pending = [task for task in fixed_pending if task["config"]["provider"] != "Google"]
print(
    f"{len(fixed_tasks):,} fixed logical batches; "
    f"{len(fixed_tasks) - len(fixed_pending):,} complete; "
    f"{len(fixed_standard_pending):,} standard and {len(fixed_google_pending):,} Gemini pending."
)


1,500 fixed logical batches; 1,500 complete; 0 standard and 0 Gemini pending.


### Submit or resume fixed Gemini batches

The request schedule retains the same 50 batches and prompt hashes.


In [12]:
fixed_google_job_path = RESULTS / "gemini_fixed_batch_job.json"
fixed_task_lookup = {
    task["request_key"]: task for task in fixed_tasks if task["config"]["provider"] == "Google"
}
fixed_google_job_name = None
if fixed_google_pending:
    pending_keys = {task["request_key"] for task in fixed_google_pending}
    saved_job = json.loads(fixed_google_job_path.read_text()) if fixed_google_job_path.exists() else None
    if saved_job and pending_keys.issubset(set(saved_job["request_keys"])):
        fixed_google_job_name = saved_job["name"]
        print(f"Resuming fixed Gemini batch job {fixed_google_job_name}.")
    else:
        inline_requests = []
        for task in fixed_google_pending:
            request = {
                "contents": [{"role": "user", "parts": [{"text": make_prompt(task["batch"])}]}],
                "generationConfig": {
                    "temperature": 0.1,
                    "responseMimeType": "application/json",
                    "responseJsonSchema": make_schema(len(task["batch"])),
                    "thinkingConfig": {"thinkingLevel": "LOW"},
                },
            }
            inline_requests.append({
                "request": request, "metadata": {"key": task["request_key"]}
            })
        payload = {
            "batch": {
                "display_name": f"llm-var-fixed-{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}",
                "input_config": {"requests": {"requests": inline_requests}},
            }
        }
        response = requests.post(
            "https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-pro-preview:batchGenerateContent",
            headers={
                "x-goog-api-key": os.environ["GOOGLE_API_KEY"],
                "Content-Type": "application/json",
            },
            json=payload, timeout=180,
        )
        response.raise_for_status()
        created = response.json()
        fixed_google_job_name = created["name"]
        fixed_google_job_path.write_text(json.dumps({
            "name": fixed_google_job_name,
            "request_keys": [task["request_key"] for task in fixed_google_pending],
            "created_response": created,
        }, indent=2))
        print(f"Submitted {len(fixed_google_pending):,} fixed Gemini requests as {fixed_google_job_name}.")


### Execute uncached fixed requests

Reuse the provider adapters and retry policy above.


In [13]:
fixed_failures = []
provider_workers = {"OpenAI": 40, "DeepInfra": 48}
pools = {
    provider: ThreadPoolExecutor(max_workers=workers)
    for provider, workers in provider_workers.items()
}
try:
    future_to_task = {
        pools[task["config"]["provider"]].submit(classify_task, task): task
        for task in fixed_standard_pending
    }
    for future in tqdm(
        as_completed(future_to_task), total=len(future_to_task),
        desc="Fixed OpenAI + DeepInfra batches"
    ):
        task = future_to_task[future]
        try:
            rows, audit = future.result()
            persist_fixed(rows, audit)
        except Exception as error:
            fixed_failures.append(str(error))
finally:
    for pool in pools.values():
        pool.shutdown(wait=True)



Fixed OpenAI + DeepInfra batches: 0it [00:00, ?it/s]


Fixed OpenAI + DeepInfra batches: 0it [00:00, ?it/s]

### Collect fixed Gemini responses

Each response is merged using its frozen description order.


In [14]:
if fixed_google_job_name:
    terminal_states = {
        "BATCH_STATE_SUCCEEDED", "BATCH_STATE_FAILED",
        "BATCH_STATE_CANCELLED", "BATCH_STATE_EXPIRED",
    }
    while True:
        response = requests.get(
            f"https://generativelanguage.googleapis.com/v1beta/{fixed_google_job_name}",
            headers={"x-goog-api-key": os.environ["GOOGLE_API_KEY"]},
            timeout=180,
        )
        response.raise_for_status()
        operation = response.json()
        state = operation.get("metadata", {}).get("state", operation.get("state"))
        stats = operation.get("metadata", {}).get(
            "batchStats", operation.get("batchStats", {})
        )
        print(f"Fixed Gemini batch state: {state}; stats: {stats}")
        if operation.get("done") or state in terminal_states:
            break
        time.sleep(15)

    if operation.get("error"):
        fixed_failures.append(f"Fixed Gemini batch failed: {operation['error']}")
    else:
        batch_result = operation.get("response", operation)
        response_container = (
            batch_result.get("inlinedResponses")
            or batch_result.get("output", {}).get("inlinedResponses", {})
        )
        batch_output = response_container.get("inlinedResponses", [])
        for item in batch_output:
            request_key = item.get("metadata", {}).get("key")
            task = fixed_task_lookup.get(request_key)
            if task is None or fixed_task_is_complete(task):
                continue
            if item.get("error"):
                fixed_failures.append(f"{request_key}: {item['error']}")
                continue
            try:
                raw = item["response"]
                text = raw["candidates"][0]["content"]["parts"][0]["text"]
                parsed = validate_labels(text, len(task["batch"]))
                returned_model = raw.get("modelVersion", task["config"]["model_id"])
                timestamp = datetime.now(timezone.utc).isoformat()
                rows = [
                    {
                        "description_id": row.description_id,
                        "model": task["config"]["model"], "provider": "Google",
                        "model_id": task["config"]["model_id"],
                        "returned_model": returned_model, "run": task["run"],
                        "batch_id": task["batch_id"], "position_in_batch": position,
                        "label": label, "label_value": LABEL_VALUE[label],
                        "timestamp_utc": timestamp,
                    }
                    for position, (row, label)
                    in enumerate(zip(task["batch"].itertuples(), parsed))
                ]
                prompt = make_prompt(task["batch"])
                audit = {
                    "request_key": request_key, "model": task["config"]["model"],
                    "provider": "Google", "requested_model": task["config"]["model_id"],
                    "returned_model": returned_model, "run": task["run"],
                    "batch_id": task["batch_id"], "attempt": 1,
                    "timestamp_utc": timestamp, "latency_seconds": None,
                    "description_ids": task["batch"]["description_id"].tolist(),
                    "prompt_sha256": hashlib.sha256(prompt.encode()).hexdigest(),
                    "settings": {
                        "temperature": 0.1, "reasoning": "low",
                        "batch_size": len(task["batch"]), "service": "Batch API",
                        "batch_design": "fixed",
                    },
                    "usage": raw.get("usageMetadata", {}), "raw_response": raw,
                }
                persist_fixed(rows, audit)
            except Exception as error:
                fixed_failures.append(f"{request_key}: {error}")


### Require a complete fixed-input panel

Stop on failed requests or incomplete model/run coverage before analysis.


In [15]:
if fixed_failures:
    print("\n".join(fixed_failures[:20]))
    raise RuntimeError(
        f"{len(fixed_failures)} fixed batches failed. Re-run to retry only missing batches."
    )

fixed_classifications = pd.read_csv(fixed_labels_path)
fixed_key = ["description_id", "model_id", "run"]
fixed_classifications = (
    fixed_classifications.drop_duplicates(fixed_key, keep="last")
    .sort_values(["model", "run", "description_id"])
)
fixed_classifications.to_csv(fixed_labels_path, index=False)

fixed_expected = FIXED_N * len(MODELS) * RUNS
assert len(fixed_classifications) == fixed_expected
assert fixed_classifications[fixed_key].duplicated().sum() == 0
assert set(fixed_classifications["label"]) == set(LABELS)
assert all(
    fixed_classifications.groupby(["model_id", "run", "batch_id"])["description_id"]
    .apply(list).map(tuple).groupby(level=[0, 2]).nunique().eq(1)
)
print(f"Complete: {len(fixed_classifications):,} fixed-batch classifications.")


Complete: 30,000 fixed-batch classifications.


In [16]:
fixed_audit_summary = (
    fixed_classifications.groupby(["model", "provider", "returned_model"], as_index=False)
    .agg(classifications=("description_id", "size"), runs=("run", "nunique"))
)
fixed_audit_summary["experiment"] = "fixed_batch"
fixed_audit_summary.to_csv(TABLES / "fixed_inference_audit_summary.csv", index=False)

# Verify the defining fixed-batch property from the persisted response audit: every
# model/run saw byte-identical prompt text for each batch ID.
fixed_response_records = [
    json.loads(line) for line in fixed_audit_path.read_text().splitlines() if line.strip()
]
fixed_prompt_audit = (
    pd.DataFrame(
        {
            "batch_id": [record["batch_id"] for record in fixed_response_records],
            "prompt_sha256": [record["prompt_sha256"] for record in fixed_response_records],
        }
    )
    .groupby("batch_id", as_index=False)
    .agg(logical_responses=("prompt_sha256", "size"), prompt_hashes=("prompt_sha256", "nunique"), prompt_sha256=("prompt_sha256", "first"))
)
assert len(fixed_prompt_audit) == fixed_manifest["batch_id"].nunique()
assert fixed_prompt_audit["logical_responses"].eq(len(MODELS) * RUNS).all()
assert fixed_prompt_audit["prompt_hashes"].eq(1).all()
fixed_prompt_audit.to_csv(TABLES / "fixed_batch_prompt_audit.csv", index=False)

display(fixed_audit_summary)
display(fixed_prompt_audit.head())


,model,provider,returned_model,classifications,runs,experiment
0,DeepSeek V4 Pro,DeepInfra,deepseek-ai/DeepSeek-V4-Pro,5000,5,fixed_batch
1,GLM-5.2,DeepInfra,zai-org/GLM-5.2,5000,5,fixed_batch
2,GPT-5.6 Sol,OpenAI,gpt-5.6-sol,5000,5,fixed_batch
3,Gemini 3.1 Pro,Google,gemini-3.1-pro-preview,5000,5,fixed_batch
4,Kimi K2.6,DeepInfra,moonshotai/Kimi-K2.6,5000,5,fixed_batch
5,Qwen3.5 397B,DeepInfra,Qwen/Qwen3.5-397B-A17B,5000,5,fixed_batch


,batch_id,logical_responses,prompt_hashes,prompt_sha256
0,0,30,1,5bd5d938cd0e8d684e1cf191f07b8aba30663f244862f4...
1,1,30,1,1504f660145a34b16bff8836307c2f6662e99cb770ff61...
2,2,30,1,03896604a4bf3c6162d28388c662fa362809324b777f09...
3,3,30,1,da293331f36a4c7d16eb8543ae9dc4df6bf120cb34f25b...
4,4,30,1,3701568af5dcd53a71af76df9ce88b6da72801e6055aa2...


In [17]:
# Small audit summary; detailed behavior is analyzed in Notebook 2.
audit_summary = (
    classifications.groupby(["model", "provider", "returned_model"], as_index=False)
    .agg(classifications=("description_id", "size"), runs=("run", "nunique"))
)
audit_summary.to_csv(TABLES / "inference_audit_summary.csv", index=False)
display(audit_summary)


,model,provider,returned_model,classifications,runs
0,DeepSeek V4 Pro,DeepInfra,deepseek-ai/DeepSeek-V4-Pro,23450,5
1,GLM-5.2,DeepInfra,zai-org/GLM-5.2,23450,5
2,GPT-5.6 Sol,OpenAI,gpt-5.6-sol,23450,5
3,Gemini 3.1 Pro,Google,gemini-3.1-pro-preview,23450,5
4,Kimi K2.6,DeepInfra,moonshotai/Kimi-K2.6,23450,5
5,Qwen3.5 397B,DeepInfra,Qwen/Qwen3.5-397B-A17B,23450,5
